# D086 · SQLite Transactions

Atomic e-commerce operations, rollback, savepoints, ACID, and SQLite locking.

## Why OLTP needs transactions

Placing an order changes several facts:

1. Create the order
2. Reduce inventory
3. Create the invoice

These changes must succeed together or fail together. A half-completed order corrupts operational data.

## ACID

| Property | Meaning in this example |
|---|---|
| **Atomicity** | Order, inventory, and invoice changes all commit or all roll back |
| **Consistency** | Constraints and transaction logic keep stock and amounts valid |
| **Isolation** | Other connections do not observe uncommitted partial work |
| **Durability** | After commit, SQLite records the transaction persistently |

`BEGIN` starts a transaction, `COMMIT` makes it permanent, and `ROLLBACK` cancels it. SQLite has no `ABORT TRANSACTION` statement; rollback is the explicit way to abort a transaction.

In [ ]:
import os
import sqlite3
import tempfile
import uuid

# A temporary file lets two connections share one database for the lock demo.
database_path = os.path.join(
    tempfile.gettempdir(), f"d086_ecommerce_{uuid.uuid4().hex}.db"
)

connection = sqlite3.connect(database_path)
connection.row_factory = sqlite3.Row
connection.execute("PRAGMA foreign_keys = ON")
print("SQLite version:", sqlite3.sqlite_version)

In [ ]:
connection.executescript("""
CREATE TABLE inventory (
    product_id     INTEGER PRIMARY KEY,
    product_name   TEXT NOT NULL,
    unit_price     REAL NOT NULL CHECK (unit_price >= 0),
    stock_quantity INTEGER NOT NULL CHECK (stock_quantity >= 0)
);

CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT NOT NULL,
    product_id    INTEGER NOT NULL REFERENCES inventory(product_id),
    quantity      INTEGER NOT NULL CHECK (quantity > 0),
    order_status  TEXT NOT NULL DEFAULT 'PLACED'
);

CREATE TABLE invoices (
    invoice_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id   INTEGER NOT NULL UNIQUE REFERENCES orders(order_id),
    amount     REAL NOT NULL CHECK (amount >= 0),
    status     TEXT NOT NULL DEFAULT 'UNPAID'
);

INSERT INTO inventory VALUES
    (1, 'Mechanical Keyboard', 2499.00, 10),
    (2, 'Wireless Mouse',       899.00, 20);
""")
connection.commit()

## `BEGIN` and `COMMIT`

The stock update includes `stock_quantity >= ?`. Checking stock and reducing it in the same SQL statement avoids a check-then-update race.

In [ ]:
def place_order(db, customer_name, product_id, quantity):
    try:
        db.execute("BEGIN IMMEDIATE")

        product = db.execute("""
            SELECT unit_price
            FROM inventory
            WHERE product_id = ?
        """, (product_id,)).fetchone()
        if product is None:
            raise ValueError("Product does not exist")

        result = db.execute("""
            UPDATE inventory
            SET stock_quantity = stock_quantity - ?
            WHERE product_id = ? AND stock_quantity >= ?
        """, (quantity, product_id, quantity))
        if result.rowcount != 1:
            raise ValueError("Insufficient stock")

        order = db.execute("""
            INSERT INTO orders (customer_name, product_id, quantity)
            VALUES (?, ?, ?)
        """, (customer_name, product_id, quantity))
        order_id = order.lastrowid

        db.execute("""
            INSERT INTO invoices (order_id, amount)
            VALUES (?, ?)
        """, (order_id, product["unit_price"] * quantity))

        db.commit()
        return order_id
    except Exception:
        db.rollback()
        raise

order_id = place_order(connection, "Asha", product_id=1, quantity=2)
print("Committed order:", order_id)

In [ ]:
for table in ("inventory", "orders", "invoices"):
    rows = connection.execute(f"SELECT * FROM {table}").fetchall()
    print(table, [dict(row) for row in rows])

## Abort with `ROLLBACK`

The next order requests more stock than exists. The function raises an error and rolls back every change in the transaction.

In [ ]:
before = connection.execute(
    "SELECT stock_quantity FROM inventory WHERE product_id = 1"
).fetchone()[0]

try:
    place_order(connection, "Ravi", product_id=1, quantity=100)
except ValueError as error:
    print("Transaction aborted:", error)

after = connection.execute(
    "SELECT stock_quantity FROM inventory WHERE product_id = 1"
).fetchone()[0]

print("Stock before:", before)
print("Stock after: ", after)
print("Ravi's orders:", connection.execute(
    "SELECT COUNT(*) FROM orders WHERE customer_name = 'Ravi'"
).fetchone()[0])

## Transaction state

`connection.in_transaction` shows whether the current connection has an open transaction. Avoid leaving transactions open: they retain locks and can block other OLTP requests.

In [ ]:
print("Before BEGIN:", connection.in_transaction)
connection.execute("BEGIN")
print("After BEGIN: ", connection.in_transaction)
connection.rollback()
print("After rollback:", connection.in_transaction)

## Savepoints: partial rollback

A savepoint marks a point inside a transaction. `ROLLBACK TO` cancels later work without cancelling the whole transaction.

In [ ]:
connection.execute("BEGIN")
connection.execute("""
    UPDATE invoices SET status = 'PAID' WHERE order_id = ?
""", (order_id,))

connection.execute("SAVEPOINT optional_note")
try:
    # Simulate a failed optional step after the important payment update.
    raise RuntimeError("Notification service unavailable")
except RuntimeError as error:
    connection.execute("ROLLBACK TO optional_note")
    connection.execute("RELEASE optional_note")
    print("Optional step cancelled:", error)

connection.commit()
status = connection.execute(
    "SELECT status FROM invoices WHERE order_id = ?", (order_id,)
).fetchone()[0]
print("Committed invoice status:", status)

## SQLite transaction modes

| Command | Lock behavior |
|---|---|
| `BEGIN` / `BEGIN DEFERRED` | Starts without taking a write lock; the first statement determines what is needed |
| `BEGIN IMMEDIATE` | Reserves the database for writing immediately; another writer fails or waits |
| `BEGIN EXCLUSIVE` | Takes the strongest transaction lock; in rollback-journal mode it also blocks readers |

SQLite does not support `SELECT ... FOR UPDATE`, `LOCK TABLE`, or independent row locks. Locking coordinates access to the database file. Only one connection writes at a time, although many readers can coexist. Keep write transactions short.

## Observe writer locking

Connection A starts an immediate write transaction. Connection B can still read the last committed value, but its write cannot proceed while A owns the write lock.

In [ ]:
writer_a = sqlite3.connect(database_path, timeout=0.1)
writer_b = sqlite3.connect(database_path, timeout=0.1)

writer_a.execute("BEGIN IMMEDIATE")
writer_a.execute("""
    UPDATE inventory SET stock_quantity = stock_quantity + 5
    WHERE product_id = 2
""")

# B reads the last committed snapshot, not A's uncommitted value.
print("B reads stock:", writer_b.execute(
    "SELECT stock_quantity FROM inventory WHERE product_id = 2"
).fetchone()[0])

try:
    writer_b.execute("""
        UPDATE inventory SET stock_quantity = stock_quantity - 1
        WHERE product_id = 2
    """)
except sqlite3.OperationalError as error:
    print("B write blocked:", error)
    writer_b.rollback()

writer_a.commit()

# After A commits, B can write.
writer_b.execute("""
    UPDATE inventory SET stock_quantity = stock_quantity - 1
    WHERE product_id = 2
""")
writer_b.commit()
print("B write committed after A released the lock")

## Busy timeout and WAL

- `timeout=5` or `PRAGMA busy_timeout = 5000` makes a connection wait briefly for a lock instead of failing immediately.
- `PRAGMA journal_mode = WAL` uses a write-ahead log. WAL usually improves reader/writer concurrency because readers can continue while a writer commits.
- WAL still permits only **one writer at a time**.
- Long transactions reduce concurrency regardless of journal mode.

Locking protects isolation, but application logic must still use conditions, constraints, and transactions correctly.

In [ ]:
print("Journal mode:", connection.execute("PRAGMA journal_mode").fetchone()[0])
print("Busy timeout:", connection.execute("PRAGMA busy_timeout").fetchone()[0], "ms")
print("Final mouse stock:", connection.execute(
    "SELECT stock_quantity FROM inventory WHERE product_id = 2"
).fetchone()[0])

## Transaction rules for OLTP

1. Put all changes for one business event in one transaction.
2. Commit only after every required operation succeeds.
3. Roll back on every error path.
4. Keep transactions short; never wait for user input inside one.
5. Use constraints and conditional updates, not only application checks.
6. Expect lock contention and configure a sensible busy timeout.
7. Do not treat generated IDs or gaps as proof that a transaction committed.

In [ ]:
writer_a.close()
writer_b.close()
connection.close()

os.remove(database_path)
print("Connections closed and temporary database removed.")